<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 03 - Técnicas de Pré-Processamento de Texto e Similaridade</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2> Técnicas para Pré-Processamento - Parte 2</h2>

<p>Uma vez que o texto já foi devidamente tratado, removendo stopwords e pontuações, e aplicando stemming ou lemmatization, agora precisamos contar a frequência das palavras (ou n-grams) que utilizaremos em seguida como atributos para as técnicas de aprendizado de máquina.</p>

<b>1. TF-IDF (Term Frequency - Inverse Document Frequency)</b>

<p><b>Term Frequency:</b> um termo que aparece muito em um documento, tende a ser um termo importante. Em resumo, divide-se o número de vezes em que um termo apareceu pelo maior número de vezes em que algum outro termo apareceu no documento.</p>

tf<sub>wd</sub> = f<sub>wd</sub> / m<sub>wd</sub>

onde:<br>
f<sub>wd</sub> é o número de vezes em que o termo <i>w</i> aparece no documento <i>d</i>.<br>
m<sub>wd</sub> é o maior valor de f<sub>wd</sub> obtido para algum termo do documento <i>d</i><br>

<p><b>Inverse Document Frequency:</b> um termo que aparece em poucos documentos pode ser um bom descriminante. Obtem-se dividindo o número de documentos pelo número de documentos em que o termo aparece.</p>

idf<sub>w</sub> = log<sub>2</sub>(n / n<sub>w</sub>)

onde:<br>
n é o número de documentos no corpus
n<sub>w</sub> é o número de documentos em que o termo <i>w</i> aparece.

Na prática, usa-se:

tf-idf = tf<sub>wd</sub> * idf<sub>w</sub>

Podemos calcular o TF-IDF de um corpus usando o pacote <b>scikit-learn</b>. Primeiramente, vamos abrir novamente o texto de Hamlet e armazenar as sentenças em uma ndarray do numpy (como se cada sentença fosse um documento do corpus):

In [1]:
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

print(hamlet_np.shape)


(2355,)


In [2]:
#hamlet_np

array(['[The Tragedie of Hamlet by William Shakespeare 1599]\n\n\nActus Primus.',
       'Scoena Prima.', 'Enter Barnardo and Francisco two Centinels.',
       ...,
       'Marching: after the which, a Peale of Ordenance are shot\noff.',
       'FINIS.', 'The tragedie of HAMLET, Prince of Denmarke.'],
      dtype='<U814')

<p>Agora vamos definir uma função para tokenização pelo scikit-learn.</p>

In [3]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer(doc):
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas
    
    

E essa função será chamada pelo objeto TfidfVectorizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

tfidf_vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)

tfs = tfidf_vectorizer.fit_transform(hamlet_np)

print(tfs.shape)

(2355, 4305)


In [5]:
print([k for k in tfidf_vectorizer.vocabulary_.keys()][:20])

['tragedie', 'hamlet', 'william', 'shakespeare', '1599', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinels', "'s", 'fran', 'nay', 'answer', 'stand', 'vnfold']


In [6]:
print(tfs[:50,:50])#linha 0 coluna 17

  (0, 17)	0.4098405429500896
  (4, 5)	1.0
  (12, 6)	0.25569565102708125
  (13, 6)	0.24183188573710981
  (22, 5)	0.5865767987601467
  (27, 0)	0.226773205328385
  (29, 5)	0.28458175550618814
  (37, 5)	0.26402019659267234
  (37, 0)	0.23602778459784524
  (39, 6)	0.13875455587670743
  (40, 13)	0.35297267044370917
  (41, 25)	0.31475626155215114
  (43, 5)	0.14416024946136125
  (46, 5)	0.32299630350190056


<b>2. TF-IDF de N-gramas</b>

Opcionalmente, podemos obter os atributos tf-idf de n-grams, combinando as classes CountVectorizer e TfidfTransformer. Em nosso exemplo, vamos utilizar apenas trigramas:

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

count_vect = CountVectorizer(ngram_range=(3,3))

n_gram_counts = count_vect.fit_transform(hamlet_np)

tfidf_transformer = TfidfTransformer()

tfs_ngrams = tfidf_transformer.fit_transform(n_gram_counts)

print(tfs_ngrams.shape)

(2355, 23483)


<b>3. Redução de Dimensionalidade</b>

<p>A transformação do corpus em atributos contendo as frequências TF-IDF em geral resultará numa ndarray bastante esparsa, ou seja, com muitas dimensões. Porém, além de isso tornar o treinamento de algoritmos mais demorado e custoso (computacionalmente falando), muitas dessas dimensões provavelmente são pouco representativas ou mesmo podem causar ruído durante o treinamento. Para resolver esse problema, podemos aplicar uma técnica de redução de dimensionalidade simples chamada <b>Singular Value Decomposition (SVD)</b>. 

<p>Essa técnica transformará os vetores da matriz original, rotacionando e escalando-os, resultando em novas representações. A redução de dimensionalidade é feita ao manter apenas as <i>k</i> dimensões mais representativas que escolhermos. Outra vantagem dessa técnica é que as dimensões originais são, de certa forma, "combinadas", o que resulta em uma nova forma de representar a combinação de termos. No contexto de PLN, essa técnica é conhecida como <b>Latent Semantic Analysis (LSA)</b></p>

In [8]:
from sklearn.decomposition import TruncatedSVD

svd_transformer = TruncatedSVD(n_components=1000)

svd_transformer.fit(tfs)

print(sorted(svd_transformer.explained_variance_ratio_)[::-1][:30])#lista de variância dos componentes, os 30 primeiros

[0.04319143964864244, 0.018619648885290458, 0.015991826821457236, 0.015984996712032906, 0.013744333318752239, 0.01177158023826396, 0.009650213667971131, 0.00951680742496111, 0.009505050204169073, 0.00946054206412684, 0.009317453675252186, 0.009270042565308353, 0.00771140689121244, 0.006983997622753737, 0.006654688885437233, 0.006419528576316656, 0.005885302030092339, 0.005372277900304977, 0.005158171190663788, 0.004931257956440387, 0.0048675271120105, 0.004789736065754575, 0.004462492865058427, 0.004339957865187973, 0.0042969022435833054, 0.004160561728442716, 0.004039480646147072, 0.00391373336967142, 0.0038585289999736034, 0.003829828436834087]


<p>Agora vamos manter as dimensões até que a variância acumulada seja maior ou igual a 0.50.</p>

In [9]:
cummulative_variance = 0.0
k = 0
for var in sorted(svd_transformer.explained_variance_ratio_)[::-1]:
    cummulative_variance += var
    if cummulative_variance >= 0.5:
        break
    else:
        k += 1
        
print(k)#143 colunas

143


<p>Transformarmos novamente, mas desta vez com o número de k componentes que obtemos anteriormente.</p>

In [10]:
svd_transformer = TruncatedSVD(n_components=k)
svd_data = svd_transformer.fit_transform(tfs)
print(sorted(svd_transformer.explained_variance_ratio_)[::-1])

[0.043191439648677665, 0.01861964889243296, 0.015991826829357268, 0.015984996723150284, 0.01374433334054493, 0.011771580254930318, 0.009650213398864533, 0.009516807650763959, 0.00950505034867059, 0.009460542018145024, 0.009317454259735403, 0.009270042349381707, 0.007711406040305303, 0.006983997476236355, 0.006654689194886753, 0.006419526563239554, 0.005885304337716929, 0.005372277217590371, 0.005158170242544269, 0.004931257114377953, 0.004867525929332231, 0.004789736540510511, 0.004462490362880839, 0.0043399555071239595, 0.004296899332030256, 0.004160554043593279, 0.00403947713942081, 0.003913727772085646, 0.0038584986276242786, 0.0038298242447990724, 0.0037136281143644845, 0.0036633904574052195, 0.00358246965522011, 0.0035617228080339373, 0.0035351633899369473, 0.0034284250134682523, 0.0033450899906256808, 0.003242955018334173, 0.0032341335885290126, 0.0031762407996352893, 0.003122685249512447, 0.003042165574266294, 0.0029313839326818997, 0.002918256334982574, 0.0028877025289479894, 0

In [11]:
print(svd_data.shape)

(2355, 143)


In [12]:
print(svd_data)

[[ 5.55005348e-03  1.63396431e-02 -1.19475856e-03 ... -8.15157646e-04
   4.82534008e-03  1.72670284e-03]
 [-5.19487632e-13 -1.18000485e-10 -3.66109204e-11 ...  5.02776964e-04
   6.36766769e-04 -4.96109788e-04]
 [ 2.00698224e-03  7.27323146e-02 -2.78993435e-02 ...  2.58719565e-02
  -1.87039701e-02  8.65348164e-03]
 ...
 [-1.11305395e-12 -1.90186460e-10 -8.62583656e-11 ...  4.26282888e-04
  -5.85057589e-04 -1.05130614e-03]
 [-1.96445757e-12 -1.53800267e-10 -1.51953899e-10 ... -3.97049030e-04
   2.74312910e-06  1.18167192e-03]
 [ 9.56880672e-03  3.02958507e-02 -7.31958921e-04 ... -1.60077717e-02
   1.98608732e-03  2.87841442e-02]]


<h2> Similaridade de Cosseno</h2>

<p>Uma vez que dois documentos são representados como vetores numéricos, é possível comparar a similaridade entre os documentos ao calcular o cosseno do ângulo entre esses documentos - uma medida de distância, não de amplitude. Para isso, basta resolver a equação do produto escalar entre os vetores, para encontrar o cosseno.</p>

<img src="http://s0.wp.com/latex.php?latex=++%5Cdisplaystyle++%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D+%3D+%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%5Ccos%7B%5Ctheta%7D+%5C%5C+%5C%5C++%5Ccos%7B%5Ctheta%7D+%3D+%5Cfrac%7B%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D%7D%7B%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%7D++&amp;bg=ffffff&amp;fg=000000&amp;s=0" alt="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " title="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " class="latex">

<img class=" wp-image-2582 " title="vector_space" src="http://blog.christianperone.com/wp-content/uploads/2013/09/vector_space.png" alt="" width="504" height="378">
<p fontsize=8>Fonte: http://blog.christianperone.com</p>

In [13]:
    documents = (
    "The sky is blue",
    "The sun is bright",
    "The sun in the sky is bright",
    "We can see the shining sun, the bright sun"
    )

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
print(tfidf_matrix.shape)

(4, 11)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

query_vect = tfidf_vectorizer.transform(["The sun is red in the sky"])#ele segmenta e tokeniza automático, mas é bom criar o próprio

cosine_similarity(query_vect, tfidf_matrix)#próx de um é mais similar, é a terceira frase

array([[0.55245238, 0.59222761, 0.9467919 , 0.37986898]])

In [16]:
import numpy as np
desired_width = 320
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns',20)
pd.set_option('display.max_rows', None)

NameError: name 'pd' is not defined

<p><b>Exercício 3:</b>Escreva um chatbot que, dado uma pergunta em Inglês, encontre uma pergunta mais parecida no corpus de perguntas e respostas disponível no Kaggle (https://www.kaggle.com/rtatman/questionanswer-dataset#S08_question_answer_pairs.txt) e exiba a resposta.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv('S08_question_answer_pairs.txt', sep='\t',encoding='utf-8')
df.head(30)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 36: invalid continuation byte

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)
df.isna().sum()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df.Question)
print(tfidf_matrix.shape)

In [ ]:
def getIndexSimilarity(question):
    query_vect = tfidf_vectorizer.transform([question])
    similarity = cosine_similarity(query_vect, tfidf_matrix)
    i = np.where(similarity[0] == similarity.max())
    return i[0][0]

# Via Input

In [ ]:
qst = input('The question is: ')
#qst = Who killed lincoln? #Did Lincoln beat in John
index = getIndexSimilarity(qst)
print('The most similar question was: {} \nThe answer is: {}'.format(df.Question.iloc[index],df.Answer.iloc[index]))

# Via Hardcode

In [ ]:
qst = "Who killed Lincoln?"
index = getIndexSimilarity(qst)
print('The most similar question was: {} \nThe answer is: {}'.format(df.Question.iloc[index],df.Answer.iloc[index]))